In [4]:
import pandas as pd
import os
import re

# ==================== 配置区域 ====================
input_file = 'AllCameras_Combined.csv'  # 改成你的文件名
output_dir = '清洗后输出'
# ================================================

# 创建输出文件夹
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. 读取数据
print("正在读取文件...")
df = pd.read_csv(input_file, encoding='utf-8')
print(f"原始数据行数：{len(df)}")

# ==================== 第一步：字段瘦身 ====================
# 只保留需要的列
keep_cols = ['页面标题', '评价星级', '评价内容', '时间', '追评内容']
existing_cols = [col for col in keep_cols if col in df.columns]
df = df[existing_cols]
print(f"字段瘦身后保留的列：{existing_cols}")
print(f"当前数据行数：{len(df)}")

# ==================== 第二步：打标签 ====================

# 2.1 从页面标题提取品牌
def extract_brand(title):
    title = str(title)
    if '尼康' in title:
        return '尼康'
    elif '索尼' in title:
        return '索尼'
    elif '佳能' in title:
        return '佳能'
    elif '富士' in title:
        return '富士'
    else:
        return '其他'

df['品牌'] = df['页面标题'].apply(extract_brand)

# 2.2 从页面标题提取型号（简单版）
def extract_model(title):
    title = str(title)
    # 常见的入门机型关键词
    models = [
        'Z30',           # 尼康
        'ZV-E10',        # 索尼
        'R50',           # 佳能
        'X-M5', 'XM5',   # 富士（两种写法都照顾）
        'A6400',         # 索尼（备用）
        'Z50',           # 尼康（备用）
        'R10',           # 佳能（备用）
        'X-T30'          # 富士（备用）
    ]
    for model in models:
        if model in title:
            # 富士 X-M5 统一返回标准名称
            if model == 'X-M5' or model == 'XM5':
                return 'X-M5'
            return model
    return '其他'

df['型号'] = df['页面标题'].apply(extract_model)

# 2.3 处理评价星级（转为数字）
def star_to_number(star):
    star = str(star)
    if '5星' in star:
        return 5
    elif '4星' in star:
        return 4
    elif '3星' in star:
        return 3
    elif '2星' in star:
        return 2
    elif '1星' in star:
        return 1
    else:
        return None

df['评分'] = df['评价星级'].apply(star_to_number)

# 2.4 评分分组
df['评分分组'] = df['评分'].apply(lambda x:
    '好评' if x >= 4 else ('中评' if x == 3 else '差评') if pd.notna(x) else '未知'
)

# 2.5 评论字数
df['评价内容'] = df['评价内容'].astype(str)
df['评论字数'] = df['评价内容'].str.len()

# 2.6 处理时间（只保留日期部分）
df['时间'] = pd.to_datetime(df['时间'], errors='coerce').dt.date

print("打标签完成")

# ==================== 第三步：异常值过滤 ====================
初始行数 = len(df)

# 3.1 删除系统默认好评
df = df[~df['评价内容'].str.contains('此用户未填写评价内容', na=False)]

# 3.2 删除字数少于5的评论（太短没意义）
df = df[df['评论字数'] >= 5]

# 3.3 删除明显是AI生成的评论（可选）
# 如果你的数据里有"由AI生成"字样
df = df[~df['评价内容'].str.contains('由AI生成', na=False)]

# 3.4 删除纯标点/表情的评论
def is_valid_text(text):
    if len(text) < 5:
        return False
    # 如果内容全是标点符号，删除
    if re.match(r'^[\s\.。，,!！?？、；:：~～]+$', text):
        return False
    return True

df = df[df['评价内容'].apply(is_valid_text)]

删除行数 = 初始行数 - len(df)
print(f"过滤完成，删除 {删除行数} 行无效数据，剩余 {len(df)} 行")

# ==================== 第四步：重新排列列的顺序 ====================
# 按方便阅读的顺序排列
final_cols = ['品牌', '型号', '评分分组', '评分', '评价内容', '评论字数', '时间', '追评内容', '页面标题', '评价星级']
final_cols = [col for col in final_cols if col in df.columns]
df = df[final_cols]

# ==================== 第五步：保存文件 ====================

# 5.1 保存完整清洗版
full_path = os.path.join(output_dir, 'CameraReviews_Cleaned.xlsx')
df.to_excel(full_path, index=False)
print(f"\n完整清洗版已保存：{full_path}")

# 5.2 按品牌拆分
brands = df['品牌'].unique()
print(f"\n按品牌拆分：")
for brand in brands:
    if pd.notna(brand) and brand != '其他':
        df_brand = df[df['品牌'] == brand]
        brand_path = os.path.join(output_dir, f'{brand}_评论.csv')
        df_brand.to_csv(brand_path, index=False, encoding='utf-8-sig')
        print(f'{brand}：{len(df_brand)} 条')

# ==================== 第六步：输出统计信息 ====================
print("\n" + "="*50)
print("最终数据统计")
print("="*50)
print(f"总评论数：{len(df)}")
print("\n各品牌数量：")
print(df['品牌'].value_counts())
print("\n评分分布：")
print(df['评分分组'].value_counts())
print("\n各型号数量：")
print(df['型号'].value_counts())

print("\n✅ 全部完成！")
print(f"清洗后文件保存在：{os.path.abspath(output_dir)}")

正在读取文件...
原始数据行数：1787
字段瘦身后保留的列：['页面标题', '评价星级', '评价内容', '时间', '追评内容']
当前数据行数：1787
打标签完成
过滤完成，删除 29 行无效数据，剩余 1758 行

完整清洗版已保存：清洗后输出\CameraReviews_Cleaned.xlsx

按品牌拆分：
尼康：188 条
索尼：193 条
佳能：494 条
富士：883 条

最终数据统计
总评论数：1758

各品牌数量：
品牌
富士    883
佳能    494
索尼    193
尼康    188
Name: count, dtype: int64

评分分布：
评分分组
好评    1718
差评      32
中评       8
Name: count, dtype: int64

各型号数量：
型号
X-M5      883
R50       494
ZV-E10    193
Z30       188
Name: count, dtype: int64

✅ 全部完成！
清洗后文件保存在：C:\Users\zhazh\OneDrive\文档\学习\STAT 8017\Scripts\清洗后输出
